In [4]:
import pandas as pd
import numpy as np

# 1. Membaca file CSV
df = pd.read_csv('sales_transactions_raw.csv')

# 2. Konversi kolom order_date dengan format campuran (mixed) agar tidak error
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed')

# 3. Definisikan Fiscal Year (FY2025: 1 Sep 2024 - 31 Aug 2025, FY2026: 1 Sep 2025 - 31 Aug 2026)
def get_fiscal_year(dt):
    if pd.Timestamp('2024-09-01') <= dt <= pd.Timestamp('2025-08-31'):
        return 'FY2025'
    elif pd.Timestamp('2025-09-01') <= dt <= pd.Timestamp('2026-08-31'):
        return 'FY2026'
    else:
        return 'Outside FY'

df['fiscal_year'] = df['order_date'].apply(get_fiscal_year)
df_fy = df[df['fiscal_year'].isin(['FY2025', 'FY2026'])].copy()

# 4. Hitung Gross Profit berdasarkan kolom yang ada
df_fy['calculated_revenue'] = df_fy['qty'] * df_fy['unit_price'] * (1 - df_fy['discount'])
df_fy['gross_profit'] = df_fy['calculated_revenue'] - df_fy['cost']

# --- TASK 1: Deskriptif Analisis (Performa FY2025 vs FY2026) ---
summary_fy = df_fy.groupby('fiscal_year').agg({
    'calculated_revenue': 'sum',
    'cost': 'sum',
    'gross_profit': 'sum',
    'discount': 'mean'
}).reset_index()

summary_fy.rename(columns={'calculated_revenue': 'revenue'}, inplace=True)
summary_fy['gross_margin_%'] = (summary_fy['gross_profit'] / summary_fy['revenue']) * 100

print("="*60)
print("TASK 1: RINGKASAN KINERJA (FY2025 vs FY2026)")
print("="*60)
print(summary_fy.to_string(index=False))

# --- TASK 2: Diagnostik Analisis (Breakdown Berdasarkan Kategori Produk) ---
diagnostic_prod = df_fy.groupby(['fiscal_year', 'category']).agg({
    'calculated_revenue': 'sum',
    'gross_profit': 'sum'
}).reset_index()

diagnostic_prod.rename(columns={'calculated_revenue': 'revenue'}, inplace=True)
diagnostic_prod['margin_%'] = (diagnostic_prod['gross_profit'] / diagnostic_prod['revenue']) * 100

print("\n" + "="*60)
print("TASK 2: BREAKDOWN BERDASARKAN KATEGORI PRODUK")
print("="*60)
print(diagnostic_prod.to_string(index=False))

TASK 1: RINGKASAN KINERJA (FY2025 vs FY2026)
fiscal_year      revenue         cost  gross_profit  discount  gross_margin_%
     FY2025 2.351354e+10 2.429984e+10 -1.877373e+09  0.278823       -7.984222
     FY2026 1.527836e+10 2.092234e+10 -6.430287e+09  0.553664      -42.087558

TASK 2: BREAKDOWN BERDASARKAN KATEGORI PRODUK
fiscal_year         category      revenue  gross_profit    margin_%
     FY2025              ATK   65076750.0 -8.147500e+05   -1.251983
     FY2025         Aksesori   34777000.0  1.016570e+07   29.231101
     FY2025        Aksesoris 1871088600.0  2.529145e+08   13.516971
     FY2025               HP   85875000.0 -8.599700e+06  -10.014207
     FY2025           LAPTOP  206540000.0  2.318820e+07   11.226978
     FY2025           Laptop 9980395000.0 -2.184104e+09  -21.883939
     FY2025       Networking 1735347000.0  1.531580e+08    8.825785
     FY2025 Peralatan Kantor 2984201350.0  5.064202e+08   16.970040
     FY2025 Peralatan kantor    3105750.0 -7.379550e+06 -237.6

In [5]:
import pandas as pd
import numpy as np

# 1. Read Dataset
df = pd.read_csv('sales_transactions_raw.csv')
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed')

# 2. Standarisasi Kategori & Channel (Pembersihan Data)
category_mapping = {
    'ATK': 'ATK',
    'Aksesori': 'Aksesoris', 'Aksesoris': 'Aksesoris', 'aksesoris': 'Aksesoris',
    'HP': 'Smartphone', 'Smartphone': 'Smartphone', 'smartphone': 'Smartphone',
    'LAPTOP': 'Laptop', 'Laptop': 'Laptop', 'laptop': 'Laptop',
    'Networking': 'Networking', 'networking': 'Networking',
    'Peralatan Kantor': 'Peralatan Kantor', 'Peralatan kantor': 'Peralatan Kantor',
    'Printer & Tinta': 'Printer & Tinta', 'Printer&Tinta': 'Printer & Tinta'
}
df['category_clean'] = df['category'].map(category_mapping).fillna(df['category'])

# 3. Filter Fiscal Year
def get_fiscal_year(dt):
    if pd.Timestamp('2024-09-01') <= dt <= pd.Timestamp('2025-08-31'):
        return 'FY2025'
    elif pd.Timestamp('2025-09-01') <= dt <= pd.Timestamp('2026-08-31'):
        return 'FY2026'
    else:
        return 'Outside FY'

df['fiscal_year'] = df['order_date'].apply(get_fiscal_year)
df_fy = df[df['fiscal_year'].isin(['FY2025', 'FY2026'])].copy()

# 4. Hitung Metrik Utama
df_fy['calculated_revenue'] = df_fy['qty'] * df_fy['unit_price'] * (1 - df_fy['discount'])
df_fy['gross_profit'] = df_fy['calculated_revenue'] - df_fy['cost']

print("=== TASK 1: RINGKASAN KINERJA PROPOSIONAL (FY2025 vs FY2026) ===")
summary_fy = df_fy.groupby('fiscal_year').agg({
    'calculated_revenue': 'sum',
    'cost': 'sum',
    'gross_profit': 'sum',
    'discount': 'mean'
}).reset_index()
summary_fy['gross_margin_%'] = (summary_fy['gross_profit'] / summary_fy['calculated_revenue']) * 100
print(summary_fy.to_string(index=False))

print("\n=== TASK 2: BREAKDOWN PER KATEGORI HASIL BERSIH ===")
diag_cat = df_fy.groupby(['fiscal_year', 'category_clean']).agg({
    'calculated_revenue': 'sum',
    'gross_profit': 'sum',
    'discount': 'mean'
}).reset_index()
diag_cat['margin_%'] = (diag_cat['gross_profit'] / diag_cat['calculated_revenue']) * 100
print(diag_cat.to_string(index=False))

print("\n=== TASK 2: BREAKDOWN PER CHANNEL (ONLINE vs OFFLINE) ===")
diag_chan = df_fy.groupby(['fiscal_year', 'channel']).agg({
    'calculated_revenue': 'sum',
    'gross_profit': 'sum',
    'discount': 'mean'
}).reset_index()
diag_chan['margin_%'] = (diag_chan['gross_profit'] / diag_chan['calculated_revenue']) * 100
print(diag_chan.to_string(index=False))

=== TASK 1: RINGKASAN KINERJA PROPOSIONAL (FY2025 vs FY2026) ===
fiscal_year  calculated_revenue         cost  gross_profit  discount  gross_margin_%
     FY2025        2.351354e+10 2.429984e+10 -1.877373e+09  0.278823       -7.984222
     FY2026        1.527836e+10 2.092234e+10 -6.430287e+09  0.553664      -42.087558

=== TASK 2: BREAKDOWN PER KATEGORI HASIL BERSIH ===
fiscal_year   category_clean  calculated_revenue  gross_profit  discount    margin_%
     FY2025              ATK        6.507675e+07 -8.147500e+05  0.330556   -1.251983
     FY2025        Aksesoris        1.905866e+09  2.630802e+08  0.329323   13.803712
     FY2025           Laptop        1.046586e+10 -2.208949e+09  0.358032  -21.106246
     FY2025       Networking        1.691674e+09 -4.664860e+07  0.237364   -2.757540
     FY2025 Peralatan Kantor        2.987307e+09  4.990406e+08  0.230773   16.705367
     FY2025  Printer & Tinta        1.450071e+09  2.363400e+07  0.253599    1.629851
     FY2025       Smartphone    